## Prepare the Generation Data For SynEnergy

In [1]:
# =========================
# Standard libraries
# =========================
import os
import json
import random
import pickle
import warnings
import math
import re

from datetime import datetime

warnings.filterwarnings("ignore")


# =========================
# Data processing
# =========================
import numpy as np
import pandas as pd
import geopandas as geopd


# =========================
# Geometry / GIS
# =========================
import shapely.wkt as wkt

from shapely.geometry import LineString, Polygon, Point

import mapclassify


# =========================
# Visualization
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

from matplotlib import cm
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
)
from matplotlib.colorbar import ColorbarBase
from mpl_toolkits.axes_grid1 import make_axes_locatable



# =========================
# Display settings
# =========================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": [
        "Times New Roman",   # Prefer if installed
        "Liberation Serif",  # Common on Ubuntu / Debian
        "Nimbus Roman",      # Common with TeX / Ghostscript
        "DejaVu Serif"       # Matplotlib fallback
    ],
    "mathtext.fontset": "stix",       # Math symbols closer to Times
    "axes.unicode_minus": False,
})

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_pickle('../../Data/e2018.pkl')
df.rename(columns={'BADGE':'badge'},inplace=True)
df.rename(columns={'DATE_RD':'date'},inplace=True)
df1 = pd.read_csv("../../Data/household_data/badge_location.csv")
df = pd.merge(df,df1[['badge','tractID','blockID']],on='badge',how='left')
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df[df["date"].dt.month == 4]
df = df.dropna(subset=["blockID", "tractID"]).reset_index(drop=True)
df["blockID"] = df["blockID"].astype(int)
df["tractID"] = df["tractID"].astype(int)
df["blockID"] = df["blockID"].astype(str)
df["tractID"] = df["tractID"].astype(str)
df.index = range(df.shape[0])
df.head(2)

,badge,date,INT_VALUE_0000,INT_VALUE_0030,INT_VALUE_0100,INT_VALUE_0130,INT_VALUE_0200,INT_VALUE_0230,INT_VALUE_0300,INT_VALUE_0330,INT_VALUE_0400,INT_VALUE_0430,INT_VALUE_0500,INT_VALUE_0530,INT_VALUE_0600,INT_VALUE_0630,INT_VALUE_0700,INT_VALUE_0730,INT_VALUE_0800,INT_VALUE_0830,INT_VALUE_0900,INT_VALUE_0930,INT_VALUE_1000,INT_VALUE_1030,INT_VALUE_1100,INT_VALUE_1130,INT_VALUE_1200,INT_VALUE_1230,INT_VALUE_1300,INT_VALUE_1330,INT_VALUE_1400,INT_VALUE_1430,INT_VALUE_1500,INT_VALUE_1530,INT_VALUE_1600,INT_VALUE_1630,INT_VALUE_1700,INT_VALUE_1730,INT_VALUE_1800,INT_VALUE_1830,INT_VALUE_1900,INT_VALUE_1930,INT_VALUE_2000,INT_VALUE_2030,INT_VALUE_2100,INT_VALUE_2130,INT_VALUE_2200,INT_VALUE_2230,INT_VALUE_2300,INT_VALUE_2330,address,district,tractID,blockID
0,E162956,2018-04-01,1.62,1.45,1.44,1.59,1.18,0.75,0.95,0.80,0.79,0.75,0.76,0.77,0.89,0.78,1.16,2.04,3.90,2.18,0.80,0.34,0.31,0.13,0.14,0.26,0.22,0.38,0.14,0.29,0.11,0.14,0.16,0.15,0.26,0.14,0.17,0.17,0.19,0.28,0.24,0.17,0.17,0.14,0.22,0.30,0.14,0.13,0.15,0.31,46.0,POINT (-84.30334 30.388433),1801,18011011
1,E162956,2018-04-02,0.17,0.15,0.16,0.27,0.13,0.16,0.11,0.13,0.29,0.12,0.14,0.10,0.31,0.09,0.18,0.55,0.88,0.60,0.54,0.52,0.88,0.21,0.61,1.87,2.68,2.51,2.52,2.25,0.17,0.42,0.18,0.18,0.31,0.22,0.33,0.27,0.66,0.11,0.42,0.49,1.00,0.92,0.26,0.34,0.64,0.21,0.38,0.16,46.0,POINT (-84.30334 30.388433),1801,18011011


In [3]:
id_cols = ["badge", "date", "address", "district", "tractID", "blockID"]

value_cols = [c for c in df.columns if c.startswith("INT_VALUE_")]
# 若 df 构造时保证列顺序就是 0000→2330，可不用再 sort；否则会乱序时按时间后缀排序更安全：
value_cols = sorted(value_cols)

assert len(value_cols) == 48, len(value_cols)

# 形状 (行数, 6 个四小时段, 每段 8 个半小时点) → 在最后一维求和 → (行数, 6)
arr_4h = df[value_cols].to_numpy(dtype="float64").reshape(len(df), 6, 8).sum(axis=2)

new_cols = [
    "INT_4h_1",  # 00:00–03:30
    "INT_4h_2",
    "INT_4h_3",
    "INT_4h_4",
    "INT_4h_5",
    "INT_4h_6",
]

df1 = pd.concat([df[id_cols].reset_index(drop=True), pd.DataFrame(arr_4h, columns=new_cols)], axis=1)

In [4]:
import pandas as pd
import numpy as np

value_cols = [f"INT_4h_{i}" for i in range(1, 7)]
meta_cols = ["address", "district", "tractID", "blockID"]

base = df1.copy()
base["day"] = pd.to_datetime(base["date"], errors="coerce").dt.normalize()

# （可选）同一天同一 badge 若有多条，先按业务规则聚合：这里用最常见的“求和”，你可改成 mean/first 等。
base = (
    base.groupby(["badge", "day"], as_index=False)[value_cols + meta_cols]
    .agg({**{c: "sum" for c in value_cols}, **{c: "first" for c in meta_cols}})
)

badges = np.sort(base["badge"].unique())

# TODO: 用你的 30 天替换这里（要包含你希望对齐的那 30 个 calendar day）
day_range = pd.date_range("2018-04-01", periods=30, freq="D").normalize()

idx = pd.MultiIndex.from_product([badges, day_range], names=["badge", "day"])

wide06 = (
    base.set_index(["badge", "day"])[value_cols]
    .reindex(idx)
    .fillna(0)
)

# （可选自检）理论上每个 badge 正好 30 行 × 6 列 → 180
assert wide06.groupby(level=0).size().eq(30).all(), "badge-day 不完整，请检查 day_range"

# （badge，day）已按字典序：先 badge，再日期；对每个 badge，30 天的 6 列按行展开成 180
n_badge = badges.size
n_day = day_range.size
arr180 = wide06.to_numpy(dtype=float).reshape(n_badge, n_day * 6)

# 列名也可用 f"{day}_{k}"；这里给一个简单的 t000 ~ t179
col180 = [f"t{i:03d}" for i in range(180)]

meta = (
    base.sort_values(["badge", "day"])
        .drop_duplicates("badge")
        .set_index("badge")[meta_cols]
    .reindex(badges)  # 与 badges 对齐
)

df2 = pd.concat(
    [
        meta,
        pd.DataFrame(arr180, index=badges, columns=col180),
    ],
    axis=1,
).reset_index(names="badge")   # pandas>=2；若报错可改成 reset_index().rename(...)

In [6]:
df2.head(2)

,badge,address,district,tractID,blockID,t000,t001,t002,t003,t004,t005,t006,t007,t008,t009,t010,t011,t012,t013,t014,t015,t016,t017,t018,t019,t020,t021,t022,t023,t024,t025,t026,t027,t028,t029,t030,t031,t032,t033,t034,t035,t036,t037,t038,t039,t040,t041,t042,t043,t044,t045,t046,t047,t048,t049,t050,t051,t052,t053,t054,t055,t056,t057,t058,t059,t060,t061,t062,t063,t064,t065,t066,t067,t068,t069,t070,t071,t072,t073,t074,t075,t076,t077,t078,t079,t080,t081,t082,t083,t084,t085,t086,t087,t088,t089,t090,t091,t092,t093,t094,t095,t096,t097,t098,t099,t100,t101,t102,t103,t104,t105,t106,t107,t108,t109,t110,t111,t112,t113,t114,t115,t116,t117,t118,t119,t120,t121,t122,t123,t124,t125,t126,t127,t128,t129,t130,t131,t132,t133,t134,t135,t136,t137,t138,t139,t140,t141,t142,t143,t144,t145,t146,t147,t148,t149,t150,t151,t152,t153,t154,t155,t156,t157,t158,t159,t160,t161,t162,t163,t164,t165,t166,t167,t168,t169,t170,t171,t172,t173,t174,t175,t176,t177,t178,t179
0,E162956,46.0,POINT (-84.30334 30.388433),1801,18011011,9.78,7.94,8.06,1.59,1.62,1.56,1.28,1.78,6.11,10.91,2.81,3.91,1.96,4.22,1.05,1.51,1.63,1.59,1.16,2.58,1.26,1.04,2.86,7.67,6.41,4.57,0.91,1.15,1.18,4.98,1.19,4.23,1.93,1.74,2.02,1.21,1.1,2.37,2.68,3.30,8.56,3.61,1.60,1.26,4.21,6.80,5.70,3.20,1.31,3.92,1.14,1.02,2.55,9.35,1.15,5.06,1.15,0.91,3.95,6.77,6.25,5.05,0.96,1.04,2.48,2.4,1.14,2.80,1.05,0.93,3.17,2.45,1.18,2.76,1.10,4.89,1.58,1.75,1.38,2.25,3.10,2.36,1.59,2.17,1.37,1.12,0.00,4.26,1.76,6.66,6.78,6.42,3.25,1.26,6.15,3.39,1.18,3.64,0.93,0.96,4.56,6.69,6.46,3.88,1.11,1.05,1.20,1.98,1.42,3.35,0.99,1.15,1.21,1.72,1.52,3.27,2.31,3.92,2.18,1.38,1.32,1.26,4.13,3.48,2.08,2.65,1.30,5.33,2.43,1.34,1.46,1.58,1.45,2.33,1.35,1.17,4.57,1.84,1.37,2.72,1.03,1.05,2.12,1.71,2.40,1.68,6.29,5.85,3.06,1.74,1.42,2.59,0.94,1.09,2.35,1.78,1.22,2.90,1.28,3.19,4.04,5.91,1.26,3.10,4.09,4.24,2.39,1.64,1.44,3.57,7.00,3.50,2.77,3.25,1.84,3.09,1.60,3.02,4.02,3.37
1,E162959,36.0,POINT (-84.2839 30.493797),1602,16022007,4.90,5.98,8.11,8.72,13.98,6.91,2.26,9.44,2.91,3.40,12.16,12.24,1.87,9.42,2.98,9.00,8.51,12.48,2.51,8.26,5.55,5.29,14.53,3.80,2.25,10.58,7.19,2.91,10.57,5.64,2.21,11.96,4.57,5.43,3.31,3.40,3.4,3.19,2.12,2.73,2.81,4.02,3.98,3.69,3.29,1.99,7.18,10.22,2.02,9.97,2.60,3.49,14.57,7.99,2.01,7.83,3.14,2.74,3.29,7.33,5.48,12.00,2.90,5.98,9.41,5.3,4.84,8.98,5.38,3.40,11.27,8.16,1.86,6.13,3.02,4.93,9.57,5.40,2.12,3.88,4.44,9.51,21.20,19.58,3.14,2.94,18.22,8.57,8.51,8.84,7.56,8.83,7.40,2.69,8.57,7.11,7.33,15.20,6.83,2.37,7.55,5.74,3.03,7.75,3.83,3.00,22.17,9.22,1.90,7.51,2.34,7.33,6.71,4.05,3.14,3.22,2.44,2.21,2.41,2.47,2.50,2.16,2.34,1.99,2.31,2.69,2.71,2.30,2.97,10.60,16.61,12.42,3.08,12.63,7.11,7.98,12.58,10.95,1.72,8.22,3.26,2.71,4.64,6.90,2.04,4.81,4.09,2.87,12.98,8.62,2.09,7.91,3.50,3.01,14.98,3.78,2.42,3.39,4.49,5.30,12.82,4.96,3.35,3.31,4.19,7.71,24.75,11.19,2.55,1.95,4.19,12.97,23.02,13.75,2.32,8.19,4.14,6.97,12.82,8.22


In [41]:
df3 = df2[["badge", "tractID", "blockID"] + [f"t{i:03d}" for i in range(180)]]
df3 = df3.drop_duplicates(subset=["badge"], keep="first")
dfx = df3[[f"t{i:03d}" for i in range(24)]]
X = dfx.to_numpy(dtype=float)
X = np.expand_dims(X, axis=-1)   # 或 axis=2
# rng = np.random.default_rng(42)  # 固定随机种子可按需删掉或改掉
# idx = rng.choice(X.shape[0], size=1000, replace=False)
# X = X[idx]
X.shape

(55366, 24, 1)

In [20]:
np.save('../SynEnergy/Data/Power/Tallahassee_seq_24_sample_55366.npy',X)

In [42]:
rng = np.random.default_rng(618)  
df3.index = range(df3.shape[0])
idx = rng.choice(df3.shape[0], size=500, replace=False)
df3 = df3.iloc[idx]

In [43]:
dfx = df3[[f"t{i:03d}" for i in range(180)]]
X = dfx.to_numpy(dtype=float)

In [44]:
X.shape

(500, 180)

In [45]:
def sliding_windows(x, window, stride=1):
    """
    将形状为 (N1, 1) 或 (N1,) 的序列切成滑动窗口。
    参数
    ----
    x : array-like
        形状 (N1, 1) 或 (N1,)
    window : int
        窗口长度
    stride : int
        步长，默认 1
    返回
    ----
    out : ndarray, 形状 (N2, window, 1)
        stride=1 且 N1 >= window 时，N2 = N1 - window + 1
    """
    x = np.asarray(x)
    if x.ndim == 1:
        x = x[:, np.newaxis]
    if x.ndim != 2 or x.shape[1] != 1:
        raise ValueError("x 必须是形状 (N1, 1) 或 (N1,)")
    N1 = x.shape[0]
    if N1 < window:
        return np.empty((0, window, 1), dtype=x.dtype)
    # 每个窗口的起始下标：0, stride, 2*stride, ... 且 start + window <= N1
    starts = np.arange(0, N1 - window + 1, stride, dtype=np.intp)
    # 索引 (N2, window) → 取值后 (N2, window, 1)
    idx = starts[:, np.newaxis] + np.arange(window, dtype=np.intp)
    return x[idx]

In [49]:
window = 42  # 举例
outs = []
for i in range(X.shape[0]):
    xi = X[i].reshape(-1, 1)           # (L, 1)
    wi = sliding_windows(xi, window)    # (L - window + 1, window, 1)
    outs.append(wi)
all_windows = np.concatenate(outs, axis=0)

In [50]:
all_windows.shape

(69500, 42, 1)

In [51]:
np.save('../SynEnergy/Data/Power/Tallahassee_seq_42_sample_69500.npy',all_windows)